# Module 3 — Exercise Solutions

---

## Exercise 1: Generation Parameter Exploration — Solution

In [ ]:
!pip install -q transformers torch bitsandbytes accelerate peft

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_REPO = "jeev1992/healthcare-assistant-lora-v2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)

SYSTEM_PROMPT = "You are a helpful healthcare assistant. Provide accurate medical information and always recommend consulting a healthcare professional."

def generate(prompt, temperature=0.7, top_p=0.9, max_new_tokens=300):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True if temperature > 0 else False
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
# Task 1: Temperature comparison
question = "What are the early warning signs of a stroke?"

for temp in [0.1, 0.5, 0.7, 1.0]:
    print(f"\n{'='*60}")
    print(f"Temperature = {temp}")
    print(f"{'='*60}")
    response = generate(question, temperature=temp)
    print(response)

In [ ]:
# Task 2: Repeatability test at temperature=1.0
question = "How does metformin work for diabetes management?"

print("Running same question 3 times at temperature=1.0:\n")
for i in range(3):
    print(f"\n--- Run {i+1} ---")
    response = generate(question, temperature=1.0)
    print(response)

In [ ]:
# Task 4: top_p comparison
question = "What vaccinations are recommended for adults over 65?"

print("--- top_p=0.5 ---")
print(generate(question, temperature=0.7, top_p=0.5))
print("\n--- top_p=0.95 ---")
print(generate(question, temperature=0.7, top_p=0.95))

**Answers:**

1. At **temperature=0.1**, the output is nearly **deterministic** — the model always picks
   the highest-probability next token. Running it twice gives almost identical results.

2. At **temperature=1.0** across 3 runs, variability is **high** — the model samples from
   a much flatter distribution, so word choices, sentence structure, and even medical
   details can differ between runs.

3. For a healthcare chatbot, choose **temperature=0.1 to 0.3** because:
   - Medical information must be **consistent** — patients shouldn't get different
     answers to the same question
   - High temperature can cause **hallucinated medical facts** (e.g., inventing
     drug interactions or wrong dosages)
   - Slight temperature >0 (like 0.1) avoids degenerate repetition while
     keeping outputs factual

4. **top_p=0.5 vs 0.95:** Lower top_p restricts sampling to fewer tokens (higher
   probability ones only), making output more conservative. top_p=0.95 allows
   more variety. The effect is similar to temperature but acts as a hard cutoff
   rather than a probability rescaling. For healthcare: top_p=0.5-0.7 is safer.

---

## Exercise 2: Adapter Enable/Disable Comparison — Solution

In [ ]:
test_questions = [
    "How is pneumonia typically diagnosed?",
    "Explain the difference between Type 1 and Type 2 diabetes.",
    "What are the common side effects of ibuprofen?",
]

for q in test_questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    
    # Disable adapters for base model behavior
    model.disable_adapter_layers()
    base_output = generate(q, temperature=0.7)
    
    # Enable adapters for fine-tuned behavior
    model.enable_adapter_layers()
    ft_output = generate(q, temperature=0.7)
    
    print(f"\n--- BASE MODEL ---")
    print(base_output)
    print(f"\n--- FINE-TUNED ---")
    print(ft_output)

**Expected Comparison:**

| Question | Better Accuracy | Has Safety Disclaimer | More Detailed |
|---|---|---|---|
| Pneumonia diagnosed | similar | ft ("consult a doctor") | base |
| Type 1 vs Type 2 | similar | ft | base |
| Ibuprofen side effects | similar | ft | base |

**Key observations:**
- The base model (Qwen 1.5B) already has strong medical knowledge
- Fine-tuning primarily changed: (1) shorter/more concise format, (2) added safety disclaimers
- The base model tends to write longer, more detailed answers
- This matches our Module 4 results: accuracy similar/improved, helpfulness decreased
  (because "detailed" and "helpful" are correlated in the evaluator's judgment)

---

## Exercise 3: Inference Cost Calculator — Solution

In [ ]:
import json

# Load benchmark results to estimate token counts
with open("../module_2_colab_finetuning/results/benchmark_results_v2.json") as f:
    results = json.load(f)

CHARS_PER_TOKEN = 4  # Rough estimate for English text

input_tokens = []
output_tokens = []

for r in results:
    input_len = 50 + len(r["prompt"]) // CHARS_PER_TOKEN
    output_len = len(r["finetuned_output"]) // CHARS_PER_TOKEN
    input_tokens.append(input_len)
    output_tokens.append(output_len)

avg_input = sum(input_tokens) / len(input_tokens)
avg_output = sum(output_tokens) / len(output_tokens)
avg_total = avg_input + avg_output

print(f"Average tokens per query: {avg_total:.0f} ({avg_input:.0f} input + {avg_output:.0f} output)")

In [ ]:
# Monthly cost comparison
QUERIES_PER_DAY = 1000
DAYS_PER_MONTH = 30

deployment_options = {
    "HF Inference Endpoints (T4)": 1.30,
    "AWS SageMaker (g5.xlarge)":   1.41,
    "RunPod (A10G)":               0.36,
}

print(f"{'Option':<35} {'$/hr':>6} {'$/day':>8} {'$/month':>10}")
print("-" * 65)

for name, hourly_rate in deployment_options.items():
    daily = hourly_rate * 24
    monthly = daily * 30
    print(f"{name:<35} ${hourly_rate:>5.2f} ${daily:>7.2f} ${monthly:>9.2f}")

**Answers:**

1. **Most cost-effective:** RunPod at ~$259/month ($0.36/hr × 24 × 30)
   vs HF at ~$936/month and SageMaker at ~$1,015/month.

2. **Factors besides cost for hospital deployment:**
   - **HIPAA compliance** — RunPod may NOT be HIPAA-compliant. AWS and HF offer
     HIPAA-eligible configurations. This alone may disqualify the cheapest option.
   - **Uptime SLA** — hospitals need 99.9%+ availability. Managed services (AWS,
     HF) offer SLAs; self-hosted requires your own monitoring and failover.
   - **Data residency** — patient data may need to stay in specific regions.
   - **Audit logging** — healthcare regulations require logging who accessed what.
   - **Auto-scaling** — managed services handle traffic spikes automatically.

3. **24/7 vs auto-scaling:** For 1,000 queries/day, auto-scaling is better.
   At ~10 queries/sec, processing 1,000 queries takes ~100 seconds. Running
   a GPU 24/7 for 100 seconds of actual work wastes ~99.9% of compute.
   Auto-scaling (scale to zero when idle) could reduce costs by 10-50x.
   However, cold-start latency (loading the model) takes 30-60 seconds,
   which may be unacceptable for a chatbot. A middle ground: keep 1 instance
   warm during business hours (8am-8pm), scale to zero overnight.

4. **Break-even for self-hosting:** Self-hosting makes financial sense when
   query volume is high enough that a GPU is utilized >50% of the time,
   AND when HIPAA/compliance requirements can be met separately.